# Samstars small-area tutorial

This notebook trains a bundled samstars model from a labeled training package, keeps the full persistent training state on disk for inspection, and then segments a small inference area with that bundle.

## Inputs

This notebook covers two major steps of samstars: training a model and running inference with that model.

The training package is a 150m tile with:

- `data/training_data_strong/img.tif`
- `data/training_data_strong/chm.tif`
- `data/training_data_strong/den.tif`
- `data/training_data_strong/crowns.gpkg`

Inference uses:

- `data/img_clipped.tif`
- `data/chm_clipped.tif`
- `data/den_clipped.tif`


In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars import train, segment


## Set paths
Before starting, lets set some of the paths. Training data requires an image raster, canopy height model (chm) raster, density of first returns raster, and crown masks stored as a geopackage. Training also requires a base crowns model and a training directory to be set. Inference requires image, chm, density rasters and an inference run directory. 

In [2]:
data_dir = repo_root / "data"
training_data_dir = data_dir / "training_data_strong"

training_img = training_data_dir / "img.tif"
training_chm = training_data_dir / "chm.tif"
training_den = training_data_dir / "den.tif"
training_crowns = training_data_dir / "crowns.gpkg"

inference_img = data_dir / "img_clipped.tif"
inference_chm = data_dir / "chm_clipped.tif"
inference_den = data_dir / "den_clipped.tif"

sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_one_area_run"
model_dir = work_dir / "model"
inference_run_dir = work_dir / "segmentation_run"

print("repo_root:", repo_root)
print("training_data_dir:", training_data_dir)
print("model_dir:", model_dir)
print("inference_run_dir:", inference_run_dir)


repo_root: /Users/iosefa/repos/sam
training_data_dir: /Users/iosefa/repos/sam/data/training_data_strong
model_dir: /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model
inference_run_dir: /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run


## Set tutorial parameters

These are explicit on purpose. Change them and compare the stored persistent artifacts under `model/_training/` if you want to inspect how the pipeline behaves.


In [3]:
rgb_bands = (1, 2, 3)

device = "mps"
storage = "persistent"

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

thin_seeds = False
thin_dist = 0.8
seed_d_min = 4.0
seed_min_dist_px = 4
seed_gauss_sigma = 1
seed_min_samples = 1
seed_merge_radius = 3.0
seed_min_canopy_height = 2.0

stardist_input_key = "index"
stardist_use_aux = True
stardist_prob_thr = 0.3
stardist_nms_thr = 0.4
stardist_window_size_m = 150.0
stardist_window_overlap_m = 30.0
stardist_dedup_iou_thr = 0.6

## Train bundled model

This writes a flat model bundle at `model_dir` and keeps persistent intermediates under `model_dir/_training/`.


In [4]:
model = train(
    img=training_img,
    chm=training_chm,
    den=training_den,
    crowns=training_crowns,
    sam_base_ckpt=sam_base_ckpt,
    out=model_dir,
    storage=storage,
    rgb_bands=rgb_bands,
    device=device,
    chip_size=chip_size,
    chip_stride=chip_stride,
    sam_epochs=sam_epochs,
    stardist_epochs=stardist_epochs,
    val_fraction=val_fraction,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    thin_seeds=thin_seeds,
    thin_dist=thin_dist,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    fast=False,
)

/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model/_training/prep/scene
tile_000_000: wrote 147 training chips
Built 147 training chips in /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model/_training/training_chips
training_chips: epochs=3 (iters/epoch=147)
training_chips epoch 1/3 loss=0.4732
training_chips epoch 2/3 loss=0.2927
training_chips epoch 3/3 loss=0.2069
✓ weights → /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model/decoder.pth
seeds in chip: 486
seed prob pcts: [1.58057983e-10 3.57160246e-07 4.26971428e-06 7.71572297e-01
 9.47911978e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.59903738e-01
 9.95365798e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.18549087e-06 8.00999125e-01
 9.98758137e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.35939438e-01
 9.99251246e-01]
seed prob pcts: [1.58057983e-10 2.13432230e-07 3.45043145e-06 6.96183690e-01
 9.2087

proposals.py (105): invalid value encountered in divide
2026-04-20 18:04:41.260016: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-20 18:04:41.260050: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-20 18:04:41.260053: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-20 18:04:41.260062: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 18:04:41.260070: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


prob_s pcts: [0.00000000e+00 1.23863909e-04 3.69253945e-01 7.13592093e-01
 8.70954216e-01]
✓ wrote proposal outputs (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model/_training/stardist_training_records/train_records.json (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/model/_training/stardist_training_records/val_records.json (0 records)
epochs=10 (iters/epoch=1)
Using default values: prob_thresh=0.5, nms_thresh=0.4.
no validation records found; reusing training data as validation_data
Epoch 1/10


2026-04-20 18:04:42.583887: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


100/100 ━━━━━━━━━━━━━━━━━━━━ 28s 249ms/step - dist_dist_iou_metric: 0.2523 - dist_loss: 6.4572 - dist_relevant_mae: 6.4567 - dist_relevant_mse: 68.8226 - loss: 1.7194 - prob_kld: 0.2776 - prob_loss: 0.4280 - val_dist_dist_iou_metric: 0.3831 - val_dist_loss: 4.9192 - val_dist_relevant_mae: 4.9182 - val_dist_relevant_mse: 38.2521 - val_loss: 1.1861 - val_prob_kld: 0.0870 - val_prob_loss: 0.2023 - learning_rate: 3.0000e-04
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 243ms/step - dist_dist_iou_metric: 0.3824 - dist_loss: 4.8075 - dist_relevant_mae: 4.8067 - dist_relevant_mse: 37.7181 - loss: 1.2066 - prob_kld: 0.0951 - prob_loss: 0.2451 - val_dist_dist_iou_metric: 0.3643 - val_dist_loss: 4.7678 - val_dist_relevant_mae: 4.7671 - val_dist_relevant_mse: 38.0751 - val_loss: 1.1233 - val_prob_kld: 0.0544 - val_prob_loss: 0.1697 - learning_rate: 3.0000e-04
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 244ms/step - dist_dist_iou_metric: 0.4336 - dist_loss: 4.2766 - dist_relevant_mae: 4.2759 - dist_

## Inspect bundle and persistent training outputs


In [5]:
sorted(p.name for p in model.root.iterdir())


['_training', 'config.json', 'decoder.pth', 'manifest.json', 'weights.h5']

In [6]:
training_proposals_dir = model.root / "_training" / "training_run" / "proposals" / "rasters"
sorted(p.name for p in training_proposals_dir.glob("*.tif"))[:20]


['tile_000_000_0_chm.tif',
 'tile_000_000_0_cost.tif',
 'tile_000_000_0_logit.tif',
 'tile_000_000_0_sam_index.tif',
 'tile_000_000_0_sam_prob.tif']

## Segment the larger clipped inference area

This reuses the bundled decoder and StarDist model, prepares the inference scene internally, and writes the full persistent run state under `inference_run_dir`.


In [7]:
result = segment(
    model=model,
    out=inference_run_dir,
    storage=storage,
    img=inference_img,
    chm=inference_chm,
    den=inference_den,
    rgb_bands=rgb_bands,
    device=device,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_prob_thr=stardist_prob_thr,
    stardist_nms_thr=stardist_nms_thr,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False,
)

function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/prep/scene


bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-04-20 18:08:49.424819: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-20 18:08:49.424842: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-20 18:08:49.424846: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-20 18:08:49.424867: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 18:08:49.424876: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> p

Loading network weights from 'weights_best.h5'.
Couldn't load thresholds from 'thresholds.json', using default values. (Call 'optimize_thresholds' to change that.)
Using default values: prob_thresh=0.5, nms_thresh=0.4.
Using effective thresholds: prob_thresh=0.3, nms_thresh=0.4.
✓ wrote 3369 crown polygons to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/crowns/crowns.gpkg


## Final output


In [8]:
result.crowns


PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/crowns/crowns.gpkg')